# 06 · Subqueries and CTEs

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

| Table | Columns |
|---|---|
| `claims` (1,000) | `claim_id`, `patient_id`, `provider_id`, `date_of_service`, `cpt_code`, `billed_amount`, `paid_amount`, `claim_status`, `denial_reason` |
| `patients` (200) | `patient_id`, `patient_name`, `date_of_birth`, `insurance_plan`, `state` |
| `providers` (13) | `provider_id`, `provider_name`, `specialty`, `state` |
| `procedures` (13) | `cpt_code`, `description`, `service_type`, `standard_charge` |

> 🇪🇸 **Para Josué:** este tema ya lo sabes (lo viste el 11/11 con los 10 claims). Aquí lo usas con 1,000 claims, y el reto ya es parte de tu análisis final: **¿dónde se pierde el dinero?**
> La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


---
## Example · A subquery in the WHERE

The 5 most expensive claims **above the average** billed amount.

> 🇪🇸 **Te recuerdo:** una **subconsulta** es una consulta **dentro de otra**, entre paréntesis. SQL resuelve primero la de adentro (aquí: el promedio, 425.09) y usa ese número afuera.
> No se puede escribir `WHERE billed_amount > AVG(billed_amount)` directo: el `WHERE` mira fila por fila y `AVG` necesita todas las filas. Por eso el promedio va en su propio `SELECT`.

In [2]:
duckdb.sql("""
    SELECT claim_id, provider_id, billed_amount
    FROM claims
    WHERE billed_amount > (SELECT AVG(billed_amount) FROM claims)
    ORDER BY billed_amount DESC
    LIMIT 5
""").df()

,claim_id,provider_id,billed_amount
0,CL00756,PR002,1650
1,CL00880,PR007,1648
2,CL00451,PR011,1648
3,CL00210,PR007,1646
4,CL00763,PR011,1642


## Exercise 1 · How many claims are above average?

Show **one row** with the number of claims whose `billed_amount` is **above the average**, and how much they bill in total.

Columns: `claims_above_avg`, `total_billed`.

✔ **1 row:** 214 · $281,930.

> 🇪🇸 El mismo `WHERE` con subconsulta del ejemplo, pero en el `SELECT` pones `COUNT(*)` y `SUM(billed_amount)` (sin `ORDER BY` ni `LIMIT`: es una sola fila).
> Fíjate: solo 214 de 1,000 claims (21%) facturan **dos tercios** del dinero (281,930 de 425,085).

In [12]:
duckdb.sql("""
    SELECT 
    COUNT(*) AS claims_above_avg,
    SUM(billed_amount) AS total_billed
    FROM claims
    WHERE billed_amount > (SELECT AVG(billed_amount) FROM claims)
""").df()

,claims_above_avg,total_billed
0,214,281930.0


## Exercise 2 · Paid claims below the average payment

Among the **Paid** claims, how many were paid **less than the average payment of Paid claims**?

Columns: `paid_below_avg`.

✔ **1 row:** 604. (The average payment of Paid claims is $255.26.)

> 🇪🇸 Aquí el filtro va **en los dos lados**:
> - Afuera: `WHERE claim_status = 'Paid' AND paid_amount < (...)`.
> - Adentro, la subconsulta también lleva su propio `WHERE claim_status = 'Paid'`, para sacar el promedio **solo** de los pagados.
> Una subconsulta es una consulta completa: puede tener su `WHERE` como cualquier otra.

In [14]:
duckdb.sql("""
    SELECT
    COUNT(*) AS paid_below_avg
    FROM claims
    WHERE claim_status = 'Paid' 
    AND paid_amount < (SELECT AVG(paid_amount) FROM claims WHERE claim_status = 'Paid')
""").df()

,paid_below_avg
0,604


---
## Example · A CTE

Total billed by each provider, as a small named table, and then use it.

> 🇪🇸 **Te recuerdo:** un **CTE** (*Common Table Expression*) es una **mini tabla con nombre** que armas con `WITH nombre AS (...)` y después usas abajo como si fuera una tabla más.
> Como una hoja auxiliar de Excel: primero calculas ahí, y la hoja principal la lee.
> Afuera del CTE se usa **`WHERE`**, no `HAVING`: el CTE ya "congeló" el cálculo y `total_billed` es una columna normal.

In [ ]:
duckdb.sql("""
    WITH provider_totals AS (
        SELECT provider_id, SUM(billed_amount) AS total_billed
        FROM claims
        GROUP BY provider_id
    )
    SELECT provider_id, total_billed
    FROM provider_totals
    ORDER BY total_billed DESC
""").df()

,provider_id,total_billed
0,PR002,96838.0
1,PR011,93578.0
2,PR007,75206.0
3,PR008,32008.0
4,PR003,29421.0
5,PR009,18844.0
6,PR004,15466.0
7,PR012,12979.0
8,PR006,12921.0
9,PR005,11885.0


## Exercise 3 · Providers that bill more than $40,000

Using a CTE named `provider_totals` with the number of claims and the total billed by each provider, show only the providers that bill **more than 40,000**, from the highest to the lowest total billed.

Columns: `provider_id`, `total_claims`, `total_billed`.

✔ **3 rows:** PR002 · 85 · $96,838 / PR011 · 87 · $93,578 / PR007 · 72 · $75,206.

> 🇪🇸 El CTE del ejemplo, más `COUNT(*) AS total_claims`. Afuera: `WHERE total_billed > 40000`.
> Estos 3 proveedores facturan más que los otros 10 juntos.

In [48]:
duckdb.sql("""
    WITH provider_totals AS(
    SELECT 
    provider_id, 
    COUNT(*) as total_claims,
    SUM(billed_amount) AS total_billed 
    FROM claims
    GROUP BY provider_id
    )
    SELECT provider_id,total_claims, total_billed
    FROM provider_totals
    WHERE total_billed > 40000
    ORDER BY total_billed DESC
""").df()

,provider_id,total_claims,total_billed
0,PR002,85,96838.0
1,PR011,87,93578.0
2,PR007,72,75206.0


## Exercise 4 · Patients with 4 or more denials ⭐

Using a CTE named `patient_denials` with the number of **Denied** claims of each patient, show the patients with **4 or more** denied claims, with their name and insurance plan. Sort from the most to the fewest denials, and then by `patient_name` (A to Z).

Columns: `patient_name`, `insurance_plan`, `denied_claims`.

✔ **6 rows.** The first one is **Diego Wilson** · Medicaid · 5. The second one is **David Wilson** · Commercial · 4.

> 🇪🇸 Dos pasos:
> 1. En el CTE: `WHERE claim_status = 'Denied'`, `COUNT(*) AS denied_claims` y `GROUP BY patient_id`.
> 2. Afuera: `JOIN` del CTE con `patients` (código en común: `patient_id`) para traer el nombre y el plan, y `WHERE denied_claims >= 4`.
> **Alias:** usa `pd` para el CTE y `pa` para `patients`. Ojo con el `ORDER BY` de **dos** columnas.

In [44]:
duckdb.sql("""
    WITH patient_denials AS(
    SELECT patient_id,
    COUNT(*) AS denied_claims
    FROM claims
    WHERE claim_status = 'Denied'
    GROUP BY patient_id
    )
    SELECT pa.patient_name, pa.insurance_plan, pd.denied_claims
    FROM patient_denials as pd
    LEFT JOIN patients as pa
    ON pd.patient_id = pa.patient_id
    WHERE pd.denied_claims >= 4
    ORDER BY pd.denied_claims DESC,
    pa.patient_name ASC

    


""").df()

,patient_name,insurance_plan,denied_claims
0,Diego Wilson,Medicaid,5
1,David Wilson,Commercial,4
2,Jessica Williams,Medicaid,4
3,John Anderson,Commercial,4
4,Joseph Miller,Commercial,4
5,William Lopez,Medicaid,4


---
## Exercise 5 · Providers above the overall denial rate ⭐⭐

Show the providers whose denial rate is **higher than the overall denial rate** (17.4%), with their name, from the highest to the lowest rate.

Columns: `provider_name`, `total_claims`, `denial_rate`.

✔ **6 rows.** The first one is **Desert Sun Radiology** · 72 · **33.3**. The last one is **Golden Coast Heart Center** · 74 · **18.9**.

> 🇪🇸 **Te recuerdo** la fórmula de la tasa de rechazo:
> ```sql
> ROUND(100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*), 1) AS denial_rate
> ```
> Tres pasos:
> 1. CTE `provider_rates`: `provider_id`, `total_claims` y `denial_rate`, con `GROUP BY provider_id`.
> 2. Afuera: `JOIN` con `providers` para traer `provider_name`.
> 3. `WHERE denial_rate > (subconsulta con la tasa de **todos** los claims)`. **No escribas 17.4 a mano**: si mañana llegan claims nuevos, la tasa cambia y la subconsulta se actualiza sola.

In [46]:
duckdb.sql("""
    WITH provider_rates AS(
    SELECT provider_id, COUNT(*) AS total_claims, ROUND(100.0*SUM(CASE WHEN claim_status= 'Denied' THEN 1 ELSE 0 END)/ COUNT(*),1) AS denial_rate
    FROM claims
    GROUP BY provider_id)
    SELECT 
    pr.provider_name, pd.total_claims, pd.denial_rate
    FROM provider_rates as pd
    JOIN providers as pr
    ON pd.provider_id = pr.provider_id
    WHERE denial_rate > (SELECT 100.0 * SUM(CASE WHEN claim_status = 'Denied' THEN 1 ELSE 0 END) / COUNT(*) FROM claims)
    ORDER BY denial_rate DESC
""").df()

,provider_name,total_claims,denial_rate
0,Desert Sun Radiology,72,33.3
1,Harbor Rehab & Therapy,108,27.8
2,Empire Diagnostic Imaging,87,25.3
3,Lakeside Imaging,85,23.5
4,Bayview Physical Therapy,89,20.2
5,Golden Coast Heart Center,74,18.9


---
## Challenge ⭐⭐⭐ · Where is the denied money?

For the **Denied** claims, show for each `denial_reason`: how many claims, how much money they billed, and what **percentage of all the denied money** that is. Sort from the most to the least money.

Columns: `denial_reason`, `denied_claims`, `denied_amount`, `pct_of_denied_amount`.

✔ **5 rows.** The first one is **Missing prior authorization** · 70 · $61,028 · **62.6**. The last one is **Duplicate claim** · 10 · $2,626 · **2.7**.

> 🇪🇸 **Lo nuevo:** la subconsulta también puede ir **dentro del `SELECT`**, para dividir entre un total:
> ```sql
> ROUND(100.0 * SUM(billed_amount) / (SELECT SUM(billed_amount) FROM claims WHERE claim_status = 'Denied'), 1)
> ```
> Arriba: el dinero de **ese** motivo. Abajo (la subconsulta): el dinero de **todos** los rechazados. Es igual que el "% del total" de una tabla dinámica de Excel.
> El resto: `WHERE claim_status = 'Denied'` y `GROUP BY denial_reason`.
> **Este es el punto 2 de tu análisis final:** casi 2 de cada 3 dólares rechazados son por falta de autorización previa.

In [47]:
duckdb.sql("""
    SELECT denial_reason,
           COUNT(*) AS denied_claims,
           SUM(billed_amount) AS denied_amount,
           ROUND(100.0 * SUM(billed_amount) / (SELECT SUM(billed_amount) FROM claims WHERE claim_status = 'Denied'), 1) AS pct_of_denied_amount
    FROM claims
    WHERE claim_status = 'Denied'
    GROUP BY denial_reason
    ORDER BY denied_amount DESC
""").df()

,denial_reason,denied_claims,denied_amount,pct_of_denied_amount
0,Missing prior authorization,70,61028.0,62.6
1,Service not covered,42,20038.0,20.6
2,Patient not eligible,30,9513.0,9.8
3,Missing information,22,4295.0,4.4
4,Duplicate claim,10,2626.0,2.7


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. En el ejercicio 5, ¿por qué es mejor la subconsulta que escribir `WHERE denial_rate > 17.4`?
> 2. En el ejercicio 3, ¿por qué afuera del CTE se usa `WHERE total_billed > 40000` y no `HAVING`?